<a href="https://colab.research.google.com/github/aayushrouniyarcs24-cmd/AI-Lab-1BF24CS010/blob/main/1BF24CS010_Week7_FOL_ForwardReasoning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import re

# -----------------------------------------
# 1. PARSE LOGICAL EXPRESSIONS
# -----------------------------------------

def parse_atom(text):
    text = text.strip()
    match = re.fullmatch(r'(\w+)\((.*?)\)', text)

    if not match:
        raise ValueError("Invalid expression: " + text)

    predicate = match.group(1)
    args = [x.strip() for x in match.group(2).split(",")]

    return (predicate, *args)


def parse_rule(text):
    left, right = text.split("->")

    premises = [
        parse_atom(x)
        for x in re.split(r'\s*&\s*|\s+AND\s+', left.strip())
    ]

    conclusion = parse_atom(right.strip())

    return premises, conclusion


# -----------------------------------------
# 2. SUBSTITUTION
# -----------------------------------------

def subst(theta, term):
    if isinstance(term, tuple):
        return tuple(subst(theta, x) for x in term)

    while term in theta:
        term = theta[term]

    return term


# -----------------------------------------
# 3. UNIFICATION
# -----------------------------------------

def is_variable(x):
    return x.startswith("?") or (
        len(x) == 1 and x.islower()
    )


def unify(x, y, theta):
    if theta is None:
        return None

    x = subst(theta, x)
    y = subst(theta, y)

    if x == y:
        return theta

    if isinstance(x, str) and is_variable(x):
        theta[x] = y
        return theta

    if isinstance(y, str) and is_variable(y):
        theta[y] = x
        return theta

    if isinstance(x, tuple) and isinstance(y, tuple):
        if len(x) != len(y):
            return None

        for a, b in zip(x, y):
            theta = unify(a, b, theta)

            if theta is None:
                return None

        return theta

    return None


# -----------------------------------------
# 4. MATCH RULE PREMISES WITH FACTS
# -----------------------------------------

def match_premises(premises, facts, theta=None):
    if theta is None:
        theta = {}

    if not premises:
        yield theta
        return

    first = subst(theta, premises[0])

    for fact in facts:
        new_theta = unify(first, fact, theta.copy())

        if new_theta is not None:
            yield from match_premises(
                premises[1:],
                facts,
                new_theta
            )


# -----------------------------------------
# 5. FORWARD CHAINING
# -----------------------------------------

def forward_chaining(facts, rules, query):

    iteration = 0

    print("\n========== FORWARD CHAINING ==========")
    print("Query:", query)

    while True:
        iteration += 1
        new_facts = set()

        print(f"\nIteration {iteration}")

        for premises, conclusion in rules:

            for theta in match_premises(
                premises, facts
            ):

                result = subst(theta, conclusion)

                if result not in facts and result not in new_facts:
                    new_facts.add(result)

                    print("Rule:", premises, "->", conclusion)
                    print("Substitution:", theta)
                    print("New fact:", result)

                    answer = unify(result, query, {})

                    if answer is not None:
                        print("\nQUERY PROVED!")
                        print("Substitution:", answer)
                        return True

        if not new_facts:
            print("No new facts generated.")
            print("QUERY NOT PROVED.")
            return False

        facts.update(new_facts)


# -----------------------------------------
# 6. HARDCODED INPUT DATA (PRE-LOADED)
# -----------------------------------------

# Define the scenario as simple logic strings
input_facts = [
    "American(Robert)",
    "Enemy(A, America)",
    "Owns(A, T1)",
    "Missile(T1)"
]

input_rules = [
    "Missile(x) -> Weapon(x)",
    "Missile(x) & Owns(A, x) -> Sells(Robert, x, A)",
    "Enemy(x, America) -> Hostile(x)",
    "American(p) & Weapon(q) & Sells(p, q, r) & Hostile(r) -> Criminal(p)"
]

input_query = "Criminal(Robert)"

# Parse strings into internal tuple representation automatically
facts = set(parse_atom(f) for f in input_facts)
rules = [parse_rule(r) for r in input_rules]
query = parse_atom(input_query)

# -----------------------------------------
# 7. EXECUTION
# -----------------------------------------

print("Initial KB:")
for fact in facts:
    print(f"  {fact}")

print("\nRules:")
for rule in rules:
    print(f"  {rule[0]} -> {rule[1]}")

answer = forward_chaining(facts, rules, query)

print("\n========== FINAL RESULT ==========")
print("TRUE" if answer else "FALSE")


Initial KB:
  ('Missile', 'T1')
  ('Enemy', 'A', 'America')
  ('American', 'Robert')
  ('Owns', 'A', 'T1')

Rules:
  [('Missile', 'x')] -> ('Weapon', 'x')
  [('Missile', 'x'), ('Owns', 'A', 'x')] -> ('Sells', 'Robert', 'x', 'A')
  [('Enemy', 'x', 'America')] -> ('Hostile', 'x')
  [('American', 'p'), ('Weapon', 'q'), ('Sells', 'p', 'q', 'r'), ('Hostile', 'r')] -> ('Criminal', 'p')

========== FORWARD CHAINING ==========
Query: ('Criminal', 'Robert')

Iteration 1
Rule: [('Missile', 'x')] -> ('Weapon', 'x')
Substitution: {'x': 'T1'}
New fact: ('Weapon', 'T1')
Rule: [('Missile', 'x'), ('Owns', 'A', 'x')] -> ('Sells', 'Robert', 'x', 'A')
Substitution: {'x': 'T1'}
New fact: ('Sells', 'Robert', 'T1', 'A')
Rule: [('Enemy', 'x', 'America')] -> ('Hostile', 'x')
Substitution: {'x': 'A'}
New fact: ('Hostile', 'A')

Iteration 2
Rule: [('American', 'p'), ('Weapon', 'q'), ('Sells', 'p', 'q', 'r'), ('Hostile', 'r')] -> ('Criminal', 'p')
Substitution: {'p': 'Robert', 'q': 'T1', 'r': 'A'}
New fact: ('Cr